# Notebook 21 — One-shot SABER-held-out test audit

This notebook is the **only test-evaluation notebook** for the post-G5 paper.

The CICIoT2023 test partition was used in the earlier diagnostic study, so it is not
globally unseen in the history of the project. It was, however, excluded from SABER
graph construction, saliency development, method selection, recovery tuning, and the
G1–G5 gates. The manuscript must therefore call this a **SABER-held-out test audit**,
not a never-before-inspected test set.

Protocol:

1. load the frozen checkpoint registries produced by Notebooks 17b and 20b;
2. verify every checkpoint and selection hash;
3. evaluate every frozen model once, with no training or selection;
4. write a permanent lock and a complete model/per-class/edge audit.

Set `CONFIRM_ONE_SHOT_TEST = True` only after checking both registries.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform, hashlib
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src/saber").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError("Set SABER_REPO to the saber-ids-method repository.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repository:", REPO)
print("Device:", DEVICE)
print("Python:", sys.version.split()[0], "|", platform.platform())


In [ ]:
from src.saber.bridge_ciciot import load_bridge
from src.saber.deep_model import DeepCNN1D
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import (
    full_model_audit, action_weighted_boundary_inversion_rate, class_recall,
)
from src.saber.adapters import collect_logits, file_sha256
from src.saber.surgery import prune_cnn1d_channels

CONFIRM_ONE_SHOT_TEST = True  # CHANGE TO TRUE ONCE, AFTER REVIEWING THE REGISTRIES.

OUT = REPO / "results/saber/21_one_shot_test"
OUT.mkdir(parents=True, exist_ok=True)
LOCK = OUT / "TEST_EVALUATION_LOCK.json"
if LOCK.exists():
    raise RuntimeError(
        f"Test lock already exists at {LOCK}. Do not rerun or overwrite the test audit."
    )

TRAIN_LOADER, VAL_LOADER, TEST_LOADER, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
graph = pd.read_csv(REPO / "results/saber/14_risk_graph/asvg_edges_robust.csv")
example = next(iter(VAL_LOADER))[0][:8].to(DEVICE)
SHALLOW_TEACHER = SHALLOW_TEACHER.to(DEVICE).eval()

shallow_registry_path = REPO / "results/saber/17b_calibrated_checkpoint_freeze/shallow_frozen_model_registry.csv"
deep_registry_path = REPO / "results/saber/20b_depth_checkpoint_freeze/deep_frozen_model_registry.csv"
if not shallow_registry_path.exists() or not deep_registry_path.exists():
    raise FileNotFoundError(
        "Run Notebooks 17b and 20b first. Both checkpoint registries are required."
    )
shallow_registry = pd.read_csv(shallow_registry_path)
deep_registry = pd.read_csv(deep_registry_path)

print("Shallow frozen models:", len(shallow_registry))
print("Depth frozen models:", len(deep_registry))
display(shallow_registry[["method", "target_flops", "realised_flops", "post_awbir", "post_family_macro_f1"]])
display(deep_registry[["method", "regime", "realised_flops", "post_awbir", "post_family_macro_f1"]])
if not CONFIRM_ONE_SHOT_TEST:
    raise RuntimeError(
        "Registry review stop. Set CONFIRM_ONE_SHOT_TEST=True only when you are "
        "ready to evaluate all frozen models once."
    )


In [ ]:
def prune_map_from_csv(path):
    table = pd.read_csv(path)
    out = {}
    for layer, frame in table.groupby("module_path"):
        out[str(layer)] = sorted(frame["channel_index"].astype(int).tolist())
    return out

def load_pruned_from_registry(teacher, row, minimum_width):
    removed_path = REPO / row["removed_groups"]
    checkpoint_path = REPO / row["checkpoint"]
    if file_sha256(checkpoint_path) != row["checkpoint_sha256"]:
        raise RuntimeError(f"Checkpoint hash mismatch: {checkpoint_path}")
    prune_map = prune_map_from_csv(removed_path)
    student, _ = prune_cnn1d_channels(
        teacher, prune_map, example, minimum_remaining_per_layer=minimum_width
    )
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    student.load_state_dict(payload["state_dict"])
    return student.to(DEVICE).eval()

deep_teacher_path = REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt"
if not deep_teacher_path.exists():
    raise FileNotFoundError(deep_teacher_path)
DEEP_TEACHER = DeepCNN1D(len(CLASS_NAMES))
DEEP_TEACHER.load_state_dict(
    torch.load(deep_teacher_path, map_location="cpu", weights_only=False)["state_dict"]
)
DEEP_TEACHER = DEEP_TEACHER.to(DEVICE).eval()

@torch.no_grad()
def model_test_audit(model, teacher_logits):
    logits, labels, _ = collect_logits(model, TEST_LOADER, device=DEVICE)
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, edge = action_weighted_boundary_inversion_rate(
        teacher_logits, logits, labels, graph
    )
    audit["awbir"] = float(awbir)
    recalls = class_recall(labels, logits.argmax(axis=1), len(CLASS_NAMES))
    return logits, labels, audit, recalls, edge

shallow_teacher_logits, test_labels, _ = collect_logits(
    SHALLOW_TEACHER, TEST_LOADER, device=DEVICE
)
deep_teacher_logits, test_labels_deep, _ = collect_logits(
    DEEP_TEACHER, TEST_LOADER, device=DEVICE
)
if not np.array_equal(test_labels, test_labels_deep):
    raise RuntimeError("Architecture test labels are not aligned.")


In [ ]:
summary_rows = []
recall_rows = []
edge_rows = []

def append_model(architecture, method, variant, target, realised, model, teacher_logits, checkpoint_sha):
    _, labels, audit, recalls, edge = model_test_audit(model, teacher_logits)
    summary_rows.append({
        "architecture": architecture,
        "method": method,
        "variant": variant,
        "target_flops": target,
        "realised_flops": realised,
        "checkpoint_sha256": checkpoint_sha,
        **{key: float(value) for key, value in audit.items() if np.isscalar(value)},
    })
    for idx, value in enumerate(recalls):
        recall_rows.append({
            "architecture": architecture,
            "method": method,
            "variant": variant,
            "class_index": idx,
            "class_name": CLASS_NAMES[idx],
            "test_recall": float(value) if np.isfinite(value) else np.nan,
        })
    edge = edge.copy()
    edge.insert(0, "variant", variant)
    edge.insert(0, "method", method)
    edge.insert(0, "architecture", architecture)
    edge_rows.extend(edge.to_dict("records"))

# Dense anchors.
append_model("CNN1D-2block", "dense", "teacher", 0.0, 0.0,
             SHALLOW_TEACHER, shallow_teacher_logits, "teacher")
append_model("DeepCNN1D-4block", "dense", "teacher", 0.0, 0.0,
             DEEP_TEACHER, deep_teacher_logits, file_sha256(deep_teacher_path))

# 15 shallow full-recovery students.
for _, row in shallow_registry.iterrows():
    model = load_pruned_from_registry(SHALLOW_TEACHER, row, minimum_width=4)
    append_model(
        "CNN1D-2block", row["method"], "full_recovery",
        float(row["target_flops"]), float(row["realised_flops"]),
        model, shallow_teacher_logits, row["checkpoint_sha256"],
    )

# 10 depth students.
for _, row in deep_registry.iterrows():
    model = load_pruned_from_registry(DEEP_TEACHER, row, minimum_width=8)
    append_model(
        "DeepCNN1D-4block", row["method"], row["regime"],
        float(row["target_flops"]), float(row["realised_flops"]),
        model, deep_teacher_logits, row["checkpoint_sha256"],
    )

summary = pd.DataFrame(summary_rows)
per_class = pd.DataFrame(recall_rows)
edge_table = pd.DataFrame(edge_rows)
summary.to_csv(OUT / "test_model_summary.csv", index=False)
per_class.to_csv(OUT / "test_per_class_recall.csv", index=False)
edge_table.to_csv(OUT / "test_awbir_edge_audit.csv", index=False)
display(summary.sort_values(["architecture", "target_flops", "method", "variant"]))


In [ ]:
# Permanent lock. The lock records the exact registry and result hashes.
import datetime
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    commit = None

lock = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "git_commit": commit,
    "protocol": "SABER-held-out test evaluation; no training or model selection",
    "historical_note": (
        "The partition was used in the earlier diagnostic study, but was excluded "
        "from SABER graph, score, selection, recovery, and gate development."
    ),
    "shallow_registry_sha256": file_sha256(shallow_registry_path),
    "deep_registry_sha256": file_sha256(deep_registry_path),
    "test_model_summary_sha256": file_sha256(OUT / "test_model_summary.csv"),
    "test_per_class_recall_sha256": file_sha256(OUT / "test_per_class_recall.csv"),
    "test_awbir_edge_audit_sha256": file_sha256(OUT / "test_awbir_edge_audit.csv"),
    "n_models_including_teachers": int(len(summary)),
}
LOCK.write_text(json.dumps(lock, indent=2), encoding="utf-8")
print(json.dumps(lock, indent=2))
print("TEST AUDIT LOCKED. Do not rerun or tune from these results.")
